# 01 — Reading Error Messages

Python error messages are precise and informative — once you know how to read them.
This notebook walks through the most common error types you'll encounter in research
code, what each one means, and how to fix it.

## Learning Objectives

- Recognize and interpret the 7 most common Python error types
- Read a multi-level traceback and identify where the error occurred
- Use `try/except` to catch errors gracefully
- Fix errors in real analysis code

## Setup

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains

print("Setup complete.")

---
## How to Read a Python Error

Every Python error has the same structure:

```
Traceback (most recent call last):
  File "script.py", line 10, in <module>
    result = my_function(x)
  File "script.py", line 5, in my_function
    return x + y
ErrorType: description of the problem
```

- **Read from the bottom up**: the last line tells you what went wrong
- **The error type** is before the colon: `NameError`, `TypeError`, etc.
- **The message** after the colon describes the specific problem
- **The traceback** shows the chain of function calls that led to the error
- **The most recent call** (where the error actually happened) is near the bottom

In JavaScript you're used to stack traces — Python tracebacks are very similar.

---
## 1. NameError

**Meaning:** You referenced a variable name that doesn't exist (hasn't been defined yet, or was misspelled).

In JavaScript, accessing an undeclared variable gives you `undefined` or a ReferenceError.
In Python, it's always a `NameError`.

In [ ]:
# Demonstrating NameError safely with try/except
try:
    print(undefined_var)
except NameError as e:
    print(f"NameError caught: {e}")

# Common causes:
# 1. Typo in variable name
# 2. Using a variable before assigning it
# 3. Forgetting to import a module

# The fix: define the variable first
undefined_var = "now I'm defined"
print(undefined_var)  # Works fine

---
## 2. TypeError

**Meaning:** You used the wrong type for an operation. Python doesn't coerce types automatically like JavaScript does.

In JavaScript: `"hello" + 5` gives `"hello5"` (implicit coercion).
In Python: `"hello" + 5` raises a `TypeError`. This is usually a feature — it prevents silent bugs.

In [ ]:
try:
    result = "hello" + 5
except TypeError as e:
    print(f"TypeError caught: {e}")

# The fix: convert to the same type first
result = "hello" + str(5)
print(f"Fixed: {result}")

# Or if you want a number:
try:
    result2 = int("hello") + 5
except ValueError as e:
    print(f"Can't convert 'hello' to int: {e}")

# TypeErrors also appear when calling functions with wrong argument types:
try:
    len(42)  # len() expects a sequence, not an int
except TypeError as e:
    print(f"TypeError from len(42): {e}")

---
## 3. IndexError

**Meaning:** You tried to access a list (or other sequence) at an index that doesn't exist.

JavaScript would give you `undefined` for an out-of-bounds array access.
Python raises an `IndexError` — which actually helps you catch bugs faster.

In [ ]:
my_list = ["a", "b", "c"]  # indices: 0, 1, 2

try:
    print(my_list[10])
except IndexError as e:
    print(f"IndexError caught: {e}")
    print(f"List has {len(my_list)} items (valid indices: 0 to {len(my_list) - 1})")

# The fix: check the length first, or use a safe default
index = 10
if index < len(my_list):
    print(f"Item at {index}: {my_list[index]}")
else:
    print(f"Index {index} is out of range")

# Python negative indices count from the end — this is valid!
print(f"Last item: {my_list[-1]}")   # 'c'
print(f"Second-to-last: {my_list[-2]}")  # 'b'

---
## 4. KeyError

**Meaning:** You tried to access a dictionary key that doesn't exist.

In JavaScript, `obj.missingKey` gives you `undefined`.
In Python, `my_dict["missing_key"]` raises a `KeyError`.
Use `.get()` for safe access with a default value.

In [ ]:
my_dict = {"model": "gpt-4", "score": 0.87, "task": "summarization"}

try:
    print(my_dict["missing_key"])
except KeyError as e:
    print(f"KeyError caught: {e}")

# The fix: use .get() which returns None (or a default) instead of raising
value = my_dict.get("missing_key")       # Returns None
print(f".get() with missing key: {value}")

value_with_default = my_dict.get("missing_key", "unknown")  # Returns "unknown"
print(f".get() with default: {value_with_default}")

# Existing key still works fine
score = my_dict.get("score", 0.0)
print(f"Score: {score}")

# You can also check first with 'in'
if "flagged" in my_dict:
    print(my_dict["flagged"])
else:
    print("'flagged' key not present")

---
## 5. AttributeError

**Meaning:** You tried to call a method or access an attribute that doesn't exist on that type of object.

This often happens when you have the wrong type — e.g., you expected a string but got an integer.

In [ ]:
my_int = 42

try:
    my_int.split()  # integers don't have a .split() method — strings do
except AttributeError as e:
    print(f"AttributeError caught: {e}")

# The fix: make sure you have the right type
my_str = str(my_int)  # Convert to string first
print(f"str(42).split() — works? {my_str}")

# AttributeError also appears when you misspell a method name:
my_list = [3, 1, 2]
try:
    my_list.Sort()  # Python is case-sensitive! It's .sort(), not .Sort()
except AttributeError as e:
    print(f"Misspelled method: {e}")

my_list.sort()  # Correct — all lowercase
print(f"Sorted: {my_list}")

---
## 6. ValueError

**Meaning:** You passed the right type of argument, but the value is invalid for that operation.
The type is correct; the content is wrong.

In [ ]:
try:
    number = int("hello")  # 'hello' is a string (correct type), but not a valid integer
except ValueError as e:
    print(f"ValueError caught: {e}")

# The fix: validate before converting, or use try/except
def safe_int(value, default=0):
    """Convert to int, returning default if the value isn't a valid integer."""
    try:
        return int(value)
    except ValueError:
        return default

print(safe_int("42"))      # 42
print(safe_int("hello"))   # 0 (default)
print(safe_int("3.14"))    # 0 — note: int() can't handle float strings directly
print(safe_int("3.14") or int(float("3.14")))  # 3 — convert via float first

# ValueError also appears with unpacking:
try:
    a, b = [1, 2, 3]  # Can't unpack 3 values into 2 variables
except ValueError as e:
    print(f"Unpacking ValueError: {e}")

---
## 7. ZeroDivisionError

**Meaning:** You tried to divide by zero. This is particularly dangerous in research code —
if your denominator unexpectedly becomes zero, you might get a crash instead of silently wrong results.

In [ ]:
try:
    result = 1 / 0
except ZeroDivisionError as e:
    print(f"ZeroDivisionError caught: {e}")

# The fix: check before dividing, or handle the case explicitly
def safe_divide(numerator, denominator, default=0.0):
    """Divide, returning default if denominator is zero."""
    if denominator == 0:
        return default
    return numerator / denominator

print(safe_divide(10, 2))   # 5.0
print(safe_divide(10, 0))   # 0.0 (default)

# Real-world case: computing accuracy when results list might be empty
def compute_accuracy(results):
    if len(results) == 0:
        return None  # Or raise an error with a helpful message
    correct = sum(1 for r in results if r["correct"])
    return correct / len(results)

print(compute_accuracy([]))  # None — safe!

---
## 8. Reading a Multi-Level Traceback

Real errors often occur deep inside a function call chain. Here's how to read a traceback
when the error happens several levels deep.

In [ ]:
import traceback

def load_record(record_id, records):
    """Look up a record by ID."""
    return records[record_id]  # KeyError if record_id not in records

def compute_score(record_id, records):
    """Get the score for a record."""
    record = load_record(record_id, records)
    return record["score"]

def run_analysis(record_ids, records):
    """Compute scores for a list of record IDs."""
    scores = []
    for rid in record_ids:
        score = compute_score(rid, records)
        scores.append(score)
    return scores

records = {"r1": {"score": 0.9}, "r2": {"score": 0.75}}

try:
    # "r3" doesn't exist in records — this will raise a KeyError
    results = run_analysis(["r1", "r3"], records)
except KeyError as e:
    print("Error occurred! Here's the full traceback:")
    print()
    traceback.print_exc()
    print()
    print("Reading it from bottom to top:")
    print("  - KeyError: 'r3'                  <- The actual error")
    print("  - line in load_record              <- Where it happened")
    print("  - called from compute_score        <- Which called it")
    print("  - called from run_analysis         <- Which called that")
    print("  - called from this cell            <- The top-level call")

---
## Exercises

Now it's your turn. Fix the buggy code in each exercise.

### Exercise 1: Fix the KeyError

The code below raises a `KeyError`. Identify the error type, then fix it using `.get()` with a sensible default value.

In [ ]:
# Buggy code — run this to see the error
scores = {"model_a": 0.9, "model_b": 0.7}

try:
    score = scores["model_c"]
    print(f"Score: {score}")
except KeyError as e:
    print(f"Error: {e}")

In [ ]:
# YOUR FIX HERE
# Use .get() so that if 'model_c' is missing, you get a default value (e.g. None or 0.0)
scores = {"model_a": 0.9, "model_b": 0.7}

score = scores["model_c"]

print(f"Score: {score}")

In [ ]:
# Check: score should be None or a numeric default (not raise an error)
check_equal(score is None or isinstance(score, (int, float)), True,
            "score should be None or a number (not raise KeyError)")

### Exercise 2: Fix the TypeError

The code below raises a `TypeError`. Identify what's wrong and fix it so `total` is an integer.

In [ ]:
# Buggy code
try:
    total = "10" + 5
    print(f"Total: {total}")
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# YOUR FIX HERE
# Make total equal to 15 (the integer result of 10 + 5)

total = "10" + 5
print(f"Total: {total}")

In [ ]:
check_equal(total, 15, "total should equal 15")
check_type(total, int, "total should be an int")

### Exercise 3: Fix the Wrong Exception Type

The `try/except` block below is catching the wrong exception type, so the real error still propagates.
Identify the correct exception type and fix the `except` clause.

In [ ]:
# Buggy code — the except is catching the WRONG error type
data = [10, 20, 30]

try:
    value = data[99]          # This raises an IndexError
except ValueError:            # But we're catching ValueError — won't catch IndexError!
    print("Caught an error")

# You should see an uncaught error above (or it may silently fail depending on the runner)

In [ ]:
# YOUR FIX HERE
# Change the except clause to catch the correct error type
data = [10, 20, 30]
caught = False

try:
    value = data[99]
except ValueError:            # Fix this line!
    caught = True
    print("Caught the error correctly")

print(f"Error was caught: {caught}")

In [ ]:
check_equal(caught, True, "The error should be caught by the except clause")

---
## Why This Matters

When your analysis script crashes at 2am, you need to read the traceback and fix it fast.
Every AI research engineer spends significant time debugging data processing and analysis code.

The good news: Python's error messages are among the most readable of any programming language.
Once you internalize this set of error types, you'll be able to diagnose most problems in seconds.

In AI safety research specifically:
- A `KeyError` when accessing model outputs might mean your data schema changed
- A `TypeError` might mean you're mixing up raw strings with parsed JSON
- A `ZeroDivisionError` in an accuracy computation means your test set is empty — a critical failure

**Habit to build:** Before running a large batch job, test your code on 2-3 samples first.
Catch errors cheap before they happen at scale.

---
## Summary

| Error Type | Meaning | Common Fix |
|---|---|---|
| `NameError` | Variable doesn't exist | Define it, check spelling |
| `TypeError` | Wrong type for operation | Convert types explicitly |
| `IndexError` | List index out of bounds | Check `len()`, use negative indices carefully |
| `KeyError` | Dict key doesn't exist | Use `.get(key, default)` |
| `AttributeError` | Method doesn't exist on that type | Check the type, check spelling |
| `ValueError` | Right type, wrong value | Validate before converting |
| `ZeroDivisionError` | Divided by zero | Check denominator before dividing |

**Next:** [02 — Debugging Strategies](02_debugging_strategies.ipynb)